# Attention-Based DNN for Semantic-Importance-Aware Scheduling in IoT Networks
**Dataset:** TON_IoT (UNSW Canberra) – real IoT/IIoT network & telemetry records (public, academic use).
**Pipeline:** IoT record → importance classifier (Baseline DNN vs AB-DNN) → priority scheduler → Rayleigh-fading wireless uplink → network KPIs.

**Evaluation (as required):**
- *System / wireless view (main results):* latency (mean, 95th percentile), deadline-miss rate, throughput, goodput, spectral efficiency, BER and PER vs SNR.
- *Algorithm view (supporting):* accuracy, precision, recall, F1, confusion matrix.

> **Run with `Runtime → Restart and run all`** so every model is built *and* trained before it is evaluated.

## 1. Setup

In [ ]:
import random, warnings, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.special import erfc
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
plt.rcParams["figure.dpi"] = 110
import os, joblib
os.makedirs("results", exist_ok=True)
def save_show(name):                     # save every figure for the slides, then display it
    plt.savefig(f"results/{name}.png", dpi=150, bbox_inches="tight"); plt.show()
print("TensorFlow:", tf.__version__)

## 2. Load the TON_IoT file (CSV or Excel)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except ImportError:
    filename = "train_test_network.csv"     # used only when running outside Colab

if filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(filename)
else:
    try:
        df = pd.read_csv(filename, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(filename, low_memory=False, encoding="latin-1")

df.columns = [str(c).strip() for c in df.columns]
print("Loaded:", filename, "| shape:", df.shape)
display(df.head())
print("\nColumns:", list(df.columns))

## 3. Importance labels from the dataset's own attack categories
TON_IoT already labels every record (`type` = normal or an attack category). We map those **existing labels**
to three semantic-importance levels for network prioritisation – no labels are invented:

| Importance | TON_IoT `type` values | Why |
|---|---|---|
| LOW | normal | routine traffic |
| MEDIUM | scanning, password, xss, mitm | suspicious / early-stage activity |
| HIGH | dos, ddos, injection, backdoor, ransomware | active attacks needing immediate action |

In [ ]:
type_col = next((c for c in df.columns if c.lower() == "type"), None)
label_col = next((c for c in df.columns if c.lower() == "label"), None)
assert type_col is not None or label_col is not None, "No 'type' or 'label' column found – check the file."

IMPORTANCE_MAP = {
    "normal": "LOW",
    "scanning": "MEDIUM", "password": "MEDIUM", "xss": "MEDIUM", "mitm": "MEDIUM",
    "dos": "HIGH", "ddos": "HIGH", "injection": "HIGH", "backdoor": "HIGH", "ransomware": "HIGH",
}

if type_col is not None:
    t = df[type_col].astype(str).str.strip().str.lower()
    print("Original 'type' distribution:\n", t.value_counts(), "\n")
    unmapped = sorted(set(t.unique()) - set(IMPORTANCE_MAP))
    if unmapped:
        print("⚠️ Types not in the map (treated as MEDIUM):", unmapped)
    df["importance"] = t.map(IMPORTANCE_MAP).fillna("MEDIUM")
else:   # only a binary label exists -> 2 levels
    df["importance"] = np.where(df[label_col].astype(int) == 1, "HIGH", "LOW")

class_names = [c for c in ["LOW", "MEDIUM", "HIGH"] if c in df["importance"].unique()]
label_map = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)
print("Importance distribution:\n", df["importance"].value_counts())

## 4. Feature preparation

In [ ]:
# drop identifiers / timestamps / label leakage
DROP = {"ts", "date", "time", "src_ip", "dst_ip", "src_port", "label", "type", "importance"}   # src_port is a random ephemeral port
work = df.copy().replace("-", np.nan)

# packet size in bits for the wireless simulation (if byte counts exist)
byte_cols = [c for c in ["src_bytes", "dst_bytes"] if c in work.columns]
if byte_cols:
    payload = work[byte_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    work["_bits"] = (payload * 8).clip(256, 4096)       # one IoT packet: 32 B .. 512 B
else:
    work["_bits"] = 1024

feat_cols, cat_cols = [], []
for c in work.columns:
    if c in DROP or c.lower() in DROP or c == "_bits":
        continue
    num = pd.to_numeric(work[c], errors="coerce")
    if num.notna().mean() > 0.95:
        work[c] = num.fillna(num.median()); feat_cols.append(c)
    elif work[c].nunique() <= 30:            # low-cardinality categorical (proto, service, conn_state, ...)
        cat_cols.append(c)

# byte/packet counts are extremely heavy-tailed -> log(1+x) before scaling
num_block = work[feat_cols].astype(float)
num_block = num_block.apply(lambda col: np.log1p(col) if col.min() >= 0 else col)
X_df = pd.concat([num_block,
                  pd.get_dummies(work[cat_cols].astype(str), prefix=cat_cols, dtype=float)], axis=1)
X_df = X_df.loc[:, X_df.std() > 0]                       # remove constant columns
y_all = df["importance"].map(label_map).astype(int).to_numpy()
bits_all = work["_bits"].astype(float).to_numpy()

# remove exact duplicate records (avoid train/test leakage)
dup = X_df.assign(_y=y_all).duplicated()
X_df, y_all, bits_all = X_df[~dup].reset_index(drop=True), y_all[~dup.values], bits_all[~dup.values]

# keep at most 60k rows (stratified) so Colab trains in a few minutes
MAX_ROWS = 60000
if len(X_df) > MAX_ROWS:
    idx, _ = train_test_split(np.arange(len(X_df)), train_size=MAX_ROWS, stratify=y_all, random_state=SEED)
    X_df, y_all, bits_all = X_df.iloc[idx].reset_index(drop=True), y_all[idx], bits_all[idx]

FEATURES = list(X_df.columns)
print(f"Rows used: {len(X_df)} | features: {len(FEATURES)}")
print("Numeric:", feat_cols); print("Categorical (one-hot):", cat_cols)
print("Class counts:", dict(zip(class_names, np.bincount(y_all, minlength=NUM_CLASSES))))

## 5. Stratified 70 / 15 / 15 split and scaling (fitted on training data only)

In [ ]:
idx = np.arange(len(X_df))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=y_all, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=y_all[tmp], random_state=SEED)

scaler = StandardScaler().fit(X_df.values[tr])
Xtr, Xva, Xte = (np.clip(scaler.transform(X_df.values[i]), -10, 10).astype("float32") for i in (tr, va, te))
y_train, y_val, y_test = y_all[tr], y_all[va], y_all[te]
bits_test = bits_all[te]
N_FEAT = Xtr.shape[1]

cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {int(c): float(w) for c, w in zip(np.unique(y_train), cw)}
print("Train/Val/Test:", len(tr), len(va), len(te)); print("Class weights:", class_weight)

## 6. Models
- **Baseline DNN:** Dense 128 → Dropout 0.3 → Dense 64 → Softmax. Every feature is treated equally.
- **AB-DNN (proposed):** an **attention block** first scores every input feature for *this* record,
  `α = softmax(W₂ · tanh(W₁ x))`, re-weights the input `x̃ = N · (α ⊙ x)`, and the same DNN classifies `x̃`.
  So the network learns *which traffic features matter* for each record before deciding its importance.
  α is returned as a second output and plotted in Section 8.

In [ ]:
def build_dnn():
    inp = layers.Input(shape=(N_FEAT,))
    x = layers.Dense(128, activation="relu")(inp); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = models.Model(inp, out, name="Baseline_DNN")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def build_abdnn(att_units=64):
    inp = layers.Input(shape=(N_FEAT,))
    s = layers.Dense(att_units, activation="tanh", name="att_score")(inp)            # W1
    alpha = layers.Dense(N_FEAT, activation="softmax", name="attention")(s)       # W2 + softmax -> weights
    xw = layers.Multiply(name="reweight")([inp, alpha])
    xw = layers.Rescaling(float(N_FEAT), name="scale")(xw)                        # mean weight = 1
    x = layers.Dense(128, activation="relu")(xw); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = models.Model(inp, out, name="AB_DNN")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m, models.Model(inp, alpha)      # second model returns attention weights

## 7. Train both models (build + train in the same cell, so evaluation always uses trained weights)

In [ ]:
EPOCHS, BATCH = 30, 256
def train(model):
    es = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    return model.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=EPOCHS, batch_size=BATCH,
                     class_weight=class_weight, callbacks=[es], verbose=2)

tf.keras.utils.set_random_seed(SEED)
dnn_model = build_dnn(); dnn_model.summary(); hist_dnn = train(dnn_model)

tf.keras.utils.set_random_seed(SEED)
att_model, att_weights_model = build_abdnn(); att_model.summary(); hist_att = train(att_model)

### 7.1 Robustness check: retrain both models with 3 different seeds

In [ ]:
rob = []
for s in [1, 2, 3]:
    for name, builder in [("Baseline DNN", build_dnn), ("AB-DNN", lambda: build_abdnn()[0])]:
        tf.keras.utils.set_random_seed(s); m = builder()
        m.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=EPOCHS, batch_size=BATCH, class_weight=class_weight,
              callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)], verbose=0)
        yp = m.predict(Xte, verbose=0).argmax(1)
        rob.append({"model": name, "seed": s, "Accuracy": accuracy_score(y_test, yp),
                    "Macro F1": precision_recall_fscore_support(y_test, yp, average="macro")[2]})
rob = pd.DataFrame(rob)
display(rob.groupby("model")[["Accuracy", "Macro F1"]].agg(["mean", "std"]).round(4))

## 8. Algorithm-level evaluation (supporting metrics)

In [ ]:
results, preds = {}, {}
for name, m in [("Baseline DNN", dnn_model), ("AB-DNN", att_model)]:
    yp = np.argmax(m.predict(Xte, verbose=0), axis=1)
    p, r, f, _ = precision_recall_fscore_support(y_test, yp, average="macro", zero_division=0)
    results[name] = {"Accuracy": accuracy_score(y_test, yp), "Precision": p, "Recall": r, "F1": f}
    preds[name] = yp
    print(f"===== {name} =====")
    print(classification_report(y_test, yp, target_names=class_names, digits=4, zero_division=0))
display(pd.DataFrame(results).T.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, name in zip(axes, preds):
    cm = confusion_matrix(y_test, preds[name], labels=range(NUM_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"Confusion matrix – {name}"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); save_show("confusion_matrices")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in [("Baseline DNN", hist_dnn), ("AB-DNN", hist_att)]:
    axes[0].plot(h.history["val_accuracy"], label=name); axes[1].plot(h.history["val_loss"], label=name)
axes[0].set_title("Validation accuracy"); axes[1].set_title("Validation loss")
for a in axes: a.set_xlabel("Epoch"); a.legend()
plt.tight_layout(); save_show("training_curves")

### Attention weights: which traffic features decide importance?

In [ ]:
alpha = att_weights_model.predict(Xte, verbose=0)
rows = []
for k, c in enumerate(class_names):
    mean_a = alpha[y_test == k].mean(axis=0)
    for j in np.argsort(mean_a)[::-1][:8]:
        name = FEATURES[j] if len(FEATURES[j]) <= 28 else FEATURES[j][:26] + "…"
        rows.append({"class": c, "feature": name, "attention": mean_a[j]})
att_df = pd.DataFrame(rows)
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(6 * NUM_CLASSES, 4.2))
for ax, c in zip(np.atleast_1d(axes), class_names):
    d = att_df[att_df["class"] == c].iloc[::-1]
    ax.barh(d["feature"], d["attention"], color="#1f77b4"); ax.set_title(f"Top attended features – {c}")
    ax.set_xlabel("mean attention weight")
plt.tight_layout(); save_show("attention_weights")

## 9. Wireless system model
**Uplink:** IoT devices → base station over a block-Rayleigh-fading channel.

| Parameter | Value |
|---|---|
| Bandwidth *B* | 20 MHz |
| Slot (TTI) | 1 ms |
| Modulation | QPSK (2 bit/s/Hz) → 40,000 bits per slot |
| Packet size | from TON_IoT byte counts (32–512 B) |
| Channel | Rayleigh, per-packet SNR γ = γ̄·\|h\|² |
| BER (QPSK, given γ) | ½·erfc(√γ) |
| PER | 1 − (1 − BER)^L |
| HARQ | failed packets retransmitted (max 4 attempts) |
| Latency budget | HIGH 10 ms, MEDIUM 50 ms, LOW 200 ms |

**Schedulers compared (same traffic, same channel realisations):**
FIFO (no ML) · Baseline-DNN priority · AB-DNN priority · Oracle (true labels, upper bound).
Priority schedulers serve the queue of the *predicted* HIGH class first, then MEDIUM, then LOW (FIFO inside each).

In [ ]:
B_HZ, SLOT_S, BITS_PER_HZ = 20e6, 1e-3, 2
SLOT_CAPACITY = int(B_HZ * SLOT_S * BITS_PER_HZ)      # 40,000 bits per slot
MAX_TX = 4
DEADLINE_MS = {"LOW": 200, "MEDIUM": 50, "HIGH": 10}
deadline_slots = np.array([DEADLINE_MS[c] for c in class_names])
MAX_AGE = 500                                           # drop packets older than 500 ms

def ber_qpsk(snr_lin):          return 0.5 * erfc(np.sqrt(snr_lin))
def per_from_ber(ber, bits):    return 1 - (1 - ber) ** bits

# --- theory vs Monte-Carlo check of BER over Rayleigh fading ---
snr_db = np.arange(0, 31, 2); rng = np.random.default_rng(SEED)
ber_awgn = ber_qpsk(10 ** (snr_db / 10))
ber_ray_th = 0.5 * (1 - np.sqrt((10 ** (snr_db / 10)) / (1 + 10 ** (snr_db / 10))))
ber_ray_mc = []
for s in snr_db:
    nb = 400_000; bits = rng.integers(0, 2, nb)
    sym = (1 - 2 * bits[0::2] + 1j * (1 - 2 * bits[1::2])) / np.sqrt(2)
    h = (rng.normal(size=sym.size) + 1j * rng.normal(size=sym.size)) / np.sqrt(2)
    n = (rng.normal(size=sym.size) + 1j * rng.normal(size=sym.size)) / np.sqrt(2) * np.sqrt(1 / (2 * 10 ** (s / 10)))
    r = (h * sym + n) / h                                # coherent equalisation
    bh = np.empty(nb, int); bh[0::2] = (r.real < 0); bh[1::2] = (r.imag < 0)
    ber_ray_mc.append(np.mean(bh != bits))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].semilogy(snr_db, ber_awgn, "k--", label="QPSK AWGN (theory)")
axes[0].semilogy(snr_db, ber_ray_th, "b-", label="QPSK Rayleigh (theory)")
axes[0].semilogy(snr_db, np.maximum(ber_ray_mc, 1e-7), "ro", label="QPSK Rayleigh (simulated)")
axes[0].set_ylim(1e-6, 1); axes[0].set_xlabel("Average SNR (dB)"); axes[0].set_ylabel("BER")
axes[0].set_title("Bit Error Rate"); axes[0].grid(True, which="both", alpha=.3); axes[0].legend()

g = rng.exponential(1.0, 200_000)                       # |h|^2
for L in [256, 1024, 2048, 4096]:
    pers = [np.mean(per_from_ber(ber_qpsk(10 ** (s / 10) * g), L)) for s in snr_db]
    axes[1].semilogy(snr_db, pers, label=f"L = {L} bits")
axes[1].set_xlabel("Average SNR (dB)"); axes[1].set_ylabel("PER"); axes[1].set_title("Packet Error Rate (Rayleigh)")
axes[1].grid(True, which="both", alpha=.3); axes[1].legend()
plt.tight_layout(); save_show("ber_per_vs_snr")

## 10. Priority-scheduling simulation

In [ ]:
def simulate(pred_cls, true_cls, bits, load, snr_db, n_slots=3000, seed=SEED, fifo=False):
    # Discrete-time uplink queue. Returns network KPIs.
    rng = np.random.default_rng(seed)
    mean_bits = bits.mean()
    lam = load * SLOT_CAPACITY / mean_bits               # mean packet arrivals per slot
    snr_lin = 10 ** (snr_db / 10)
    queues = [collections.deque() for _ in range(NUM_CLASSES)] if not fifo else [collections.deque()]
    lat = [[] for _ in range(NUM_CLASSES)]
    sent_bits = good_bits = useful_bits = 0
    on_time = np.zeros(NUM_CLASSES); generated = np.zeros(NUM_CLASSES); dropped = np.zeros(NUM_CLASSES)
    tx_attempts = tx_fail = 0
    for t in range(n_slots):
        for _ in range(rng.poisson(lam)):                # new arrivals
            i = rng.integers(len(true_cls))
            pkt = [t, int(true_cls[i]), int(bits[i]), 0]  # arrival, true class, bits, tx count
            generated[pkt[1]] += 1
            queues[0 if fifo else int(pred_cls[i])].append(pkt)
        cap = SLOT_CAPACITY
        order = [0] if fifo else list(range(NUM_CLASSES - 1, -1, -1))  # HIGH first
        for q in (queues[k] for k in order):
            retry = []
            while q and cap >= q[0][2]:
                pkt = q.popleft()
                if t - pkt[0] > MAX_AGE:                  # stale -> dropped
                    dropped[pkt[1]] += 1; continue
                cap -= pkt[2]; sent_bits += pkt[2]; pkt[3] += 1; tx_attempts += 1
                gamma = snr_lin * rng.exponential(1.0)
                if rng.random() < per_from_ber(ber_qpsk(gamma), pkt[2]):   # packet error
                    tx_fail += 1
                    if pkt[3] < MAX_TX: retry.append(pkt)
                    else: dropped[pkt[1]] += 1
                    continue
                delay = t - pkt[0] + 1                    # ms (slot = 1 ms)
                lat[pkt[1]].append(delay); good_bits += pkt[2]
                if delay <= deadline_slots[pkt[1]]:
                    on_time[pkt[1]] += 1; useful_bits += pkt[2]
            q.extendleft(reversed(retry))                 # HARQ retransmissions go first next slot
    T = n_slots * SLOT_S
    hi = NUM_CLASSES - 1
    return {
        "HIGH mean latency (ms)": np.mean(lat[hi]) if lat[hi] else np.nan,
        "HIGH p95 latency (ms)": np.percentile(lat[hi], 95) if lat[hi] else np.nan,
        "HIGH deadline-miss rate": 1 - on_time[hi] / max(generated[hi], 1),
        "All deadline-miss rate": 1 - on_time.sum() / max(generated.sum(), 1),
        "Throughput (Mbps)": sent_bits / T / 1e6,
        "Goodput (Mbps)": good_bits / T / 1e6,
        "Semantic goodput (Mbps)": useful_bits / T / 1e6,   # delivered within latency budget
        "Spectral efficiency (b/s/Hz)": good_bits / T / B_HZ,
        "PER (measured)": tx_fail / max(tx_attempts, 1),
    }

SCHEDULERS = {"FIFO (no ML)": None, "Baseline DNN": preds["Baseline DNN"],
              "AB-DNN": preds["AB-DNN"], "Oracle": y_test}
def run_all(load, snr_db, seed=SEED):
    out = {}
    for name, p in SCHEDULERS.items():
        out[name] = simulate(y_test if p is None else p, y_test, bits_test, load, snr_db,
                             seed=seed, fifo=(p is None))
    return out

### 10.1 Sweep offered load (SNR fixed at 20 dB)

In [ ]:
LOADS = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
SNR_FIXED = 20
load_res = {L: run_all(L, SNR_FIXED) for L in LOADS}

def plot_vs(res, xs, metric, ax, xlabel):
    for s in SCHEDULERS:
        ax.plot(xs, [res[x][s][metric] for x in xs], marker="o", label=s)
    ax.set_xlabel(xlabel); ax.set_title(metric); ax.grid(alpha=.3)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, m in zip(axes.ravel(), ["HIGH mean latency (ms)", "HIGH p95 latency (ms)",
                                "HIGH deadline-miss rate", "Semantic goodput (Mbps)"]):
    plot_vs(load_res, LOADS, m, ax, "Offered load (fraction of channel capacity)")
axes[0, 0].legend(); plt.tight_layout(); save_show("latency_miss_goodput_vs_load")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, m in zip(axes, ["Throughput (Mbps)", "Goodput (Mbps)"]):
    plot_vs(load_res, LOADS, m, ax, "Offered load")
axes[0].legend(); plt.tight_layout(); save_show("throughput_goodput_vs_load")

### 10.2 Sweep average SNR (load fixed at 0.90)

In [ ]:
SNRS = [5, 10, 15, 20, 25, 30]
LOAD_FIXED = 0.90
snr_res = {s: run_all(LOAD_FIXED, s) for s in SNRS}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, m in zip(axes, ["Spectral efficiency (b/s/Hz)", "PER (measured)", "HIGH deadline-miss rate"]):
    plot_vs(snr_res, SNRS, m, ax, "Average SNR (dB)")
axes[1].set_yscale("log"); axes[0].legend(); plt.tight_layout(); save_show("se_per_miss_vs_snr")

## 11. Final results (load 0.90, SNR 20 dB, averaged over 5 random seeds)

In [ ]:
runs = [run_all(LOAD_FIXED, SNR_FIXED, seed=s) for s in range(5)]
final = pd.DataFrame({s: pd.DataFrame([r[s] for r in runs]).mean() for s in SCHEDULERS}).T
display(final.round(4))

fifo, ab, dn = final.loc["FIFO (no ML)"], final.loc["AB-DNN"], final.loc["Baseline DNN"]
def pct(a, b): return 100 * (a - b) / b if b else np.nan
print(f"AB-DNN vs FIFO   : HIGH mean latency {pct(ab['HIGH mean latency (ms)'], fifo['HIGH mean latency (ms)']):+.1f}% | "
      f"HIGH deadline-miss {ab['HIGH deadline-miss rate']:.3f} vs {fifo['HIGH deadline-miss rate']:.3f} | "
      f"semantic goodput {pct(ab['Semantic goodput (Mbps)'], fifo['Semantic goodput (Mbps)']):+.1f}%")
print(f"AB-DNN vs DNN    : HIGH mean latency {pct(ab['HIGH mean latency (ms)'], dn['HIGH mean latency (ms)']):+.1f}% | "
      f"HIGH deadline-miss {ab['HIGH deadline-miss rate']:.3f} vs {dn['HIGH deadline-miss rate']:.3f}")
print("\nClassification metrics (supporting):"); display(pd.DataFrame(results).T.round(4))
final.round(4).to_csv("results/network_kpis.csv"); pd.DataFrame(results).T.round(4).to_csv("results/classification_metrics.csv")

### 11.1 Improvement summary (chart for the results slide)

In [ ]:
kp = ["HIGH mean latency (ms)", "HIGH p95 latency (ms)", "HIGH deadline-miss rate", "Semantic goodput (Mbps)"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
colors = ["#7f7f7f", "#1f77b4", "#d62728", "#2ca02c"]
for ax, k in zip(axes, kp):
    vals = final[k]; bars = ax.bar(range(len(vals)), vals, color=colors)
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(["FIFO", "DNN", "AB-DNN", "Oracle"])
    ax.set_title(k); ax.bar_label(bars, fmt="%.2f", fontsize=9)
fig.suptitle(f"Network KPIs at load {LOAD_FIXED}, SNR {SNR_FIXED} dB (mean of 5 runs)", y=1.03)
plt.tight_layout(); save_show("improvement_summary")

imp = pd.DataFrame({
    "AB-DNN vs FIFO (%)": [pct(ab[k], fifo[k]) for k in kp],
    "AB-DNN vs Baseline DNN (%)": [pct(ab[k], dn[k]) for k in kp]}, index=kp).round(1)
display(imp); imp.to_csv("results/improvement_percentages.csv")

## 12. Save trained models and try new records

In [ ]:
dnn_model.save("results/baseline_dnn.keras")
att_model.save("results/ab_dnn.keras")
joblib.dump({"scaler": scaler, "features": FEATURES, "classes": class_names}, "results/preprocessing.joblib")
print("Saved:", sorted(os.listdir("results")))

# demo: classify 8 unseen test records and show the scheduling priority they receive
demo = np.random.default_rng(1).choice(len(Xte), 8, replace=False)
p = att_model.predict(Xte[demo], verbose=0)
display(pd.DataFrame({"true importance": [class_names[i] for i in y_test[demo]],
                      "AB-DNN prediction": [class_names[i] for i in p.argmax(1)],
                      "confidence": p.max(1).round(3),
                      "latency budget (ms)": [DEADLINE_MS[class_names[i]] for i in p.argmax(1)]}))

## 13. Notes for the report
- Data: TON_IoT (Moustafa et al., UNSW Canberra; IEEE Access 2020; Sustainable Cities and Society 2021). Cite the dataset papers.
- Importance levels come from the dataset's own `type` labels via the mapping in Section 3 (a documented design choice).
- The wireless channel and queue are simulated with standard textbook models (QPSK, Rayleigh fading, HARQ); the IoT records are real.
- The Oracle row is the best any classifier could achieve with this scheduler; the gap to Oracle shows the cost of misclassification.